<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Storytelling: การเดินทางด้วยระบบขนส่งสาธารณะในไทย

ชุดข้อมูล: `passengers68.csv` (ปี 2568) และ `passengers69.csv` (ปี 2569 มกราคม)  
ที่มา: กระทรวงคมนาคม — datagov.mot.go.th

In [17]:
!apt-get install -y fonts-tlwg-sarabun -qq
!fc-cache -fv -q

import matplotlib, shutil, os
shutil.rmtree(matplotlib.get_cachedir(), ignore_errors=True)
os.makedirs(matplotlib.get_cachedir(), exist_ok=True)
print('done')

E: Unable to locate package fonts-tlwg-sarabun
fc-cache: invalid option -- 'q'
usage: fc-cache [-EfrsvVh] [-y SYSROOT] [--error-on-no-fonts] [--force|--really-force] [--sysroot=SYSROOT] [--system-only] [--verbose] [--version] [--help] [dirs]
Build font information caches in [dirs]
(all directories in font configuration by default).

  -E, --error-on-no-fonts  raise an error if no fonts in a directory
  -f, --force              scan directories with apparently valid caches
  -r, --really-force       erase all existing caches, then rescan
  -s, --system-only        scan system-wide directories only
  -y, --sysroot=SYSROOT    prepend SYSROOT to all paths for scanning
  -v, --verbose            display status information while busy
  -V, --version            display font config version and exit
  -h, --help               display this help and exit
done


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

In [19]:
df68 = pd.read_csv('passengers68.csv', encoding='utf-8-sig', low_memory=False)
df69 = pd.read_csv('passengers69.csv', encoding='utf-8-sig', low_memory=False)

print('passengers68.csv:', df68.shape)
print('passengers69.csv:', df69.shape)
df68.head()

passengers68.csv: (69440, 8)
passengers69.csv: (3010, 8)


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2025,คน,"127,551"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2025,คน,"8,218"
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2025,คัน,"877,943"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2025,คัน,"932,642"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2025,คัน,"1,364,992"


### ทำความเข้าใจโครงสร้างข้อมูล

In [20]:
# Column structure
print(df68.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69440 entries, 0 to 69439
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   รูปแบบการเดินทาง   15696 non-null  object
 1   วัตถุประสงค์       15696 non-null  object
 2   สาธารณะ/ส่วนบุคคล  15696 non-null  object
 3   หน่วยงาน           15696 non-null  object
 4   ยานพาหนะ/ท่า       15696 non-null  object
 5   วันที่             15696 non-null  object
 6   หน่วย              15696 non-null  object
 7   ปริมาณ             15388 non-null  object
dtypes: object(8)
memory usage: 4.2+ MB
None


In [21]:
# Basic statistics
df68.describe()

,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
count,15696,15696,15696,15696,15696,15696,15696,15388
unique,4,3,2,13,43,365,2,12162
top,ทางอากาศ,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,ทอท.,ท่าอากาศยานภูมิภาค ขาออกประเทศ,21/10/2025,คน,0
freq,5111,5840,14236,3285,366,44,14236,583


In [22]:
# Unique values in categorical columns
for col in ['รูปแบบการเดินทาง', 'วัตถุประสงค์', 'สาธารณะ/ส่วนบุคคล']:
    print(f'{col}:', df68[col].unique())

รูปแบบการเดินทาง: ['ทางถนน' 'ทางน้ำ' 'ทางราง' 'ทางอากาศ' nan]
วัตถุประสงค์: ['การเดินทางระหว่างจังหวัด' 'การเดินทางภายในจังหวัด/กรุงเทพ'
 'การเดินทางระหว่างประเทศ' nan]
สาธารณะ/ส่วนบุคคล: ['สาธารณะ' 'ส่วนบุคคล' nan]


In [23]:
# All vehicle types
print(df68['ยานพาหนะ/ท่า'].unique())

['รถ บขส. และ รถร่วม' 'รถหมวด 3' 'รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ)'
 'รถยนต์ทุกประเภท (10 จุดสำรวจ)' 'รถยนต์เฉพาะ 4 ล้อ (ทางด่วน)'
 'รถยนต์ทุกประเภท (ทางด่วน)' 'รถหมวด 4' 'รถเมล์ ขสมก.' 'รถร่วม (หมวด 1)'
 'รถเอกชนเส้นปฏิรูป (หมวด 1)' 'เรือด่วนเจ้าพระยา' 'เรือคลองแสนแสบ'
 'เรือข้ามฟากเจ้าพระยา' 'เรือไฟฟ้าเจ้าพระยา' 'เรือภูมิภาค' 'รถไฟ'
 'รถไฟฟ้าสายสีน้ำเงิน' 'รถไฟฟ้าสายสีม่วง' 'รถไฟฟ้าสายสีเหลือง'
 'รถไฟฟ้าสายสีชมพู' 'รถไฟฟ้า ARL' 'รถไฟฟ้า BTS' 'รถไฟฟ้าสายสีแดง'
 'ท่าอากาศยานสุวรรณภูมิ' 'ท่าอากาศยานดอนเมือง' 'ท่าอากาศอื่น ๆ ของ ทอท.'
 'ท่าอากาศยานภูมิภาค' 'ท่าอากาศยานอู่ตะเภา' 'ท่าอากาศยานสมุย'
 'รถ บขส. ขาเข้าประเทศ' 'รถ บขส. ขาออกประเทศ' 'รถไฟ ขาเข้าประเทศ'
 'รถไฟ ขาออกประเทศ' 'ท่าเรือด่านชายแดน ขาเข้าประเทศ'
 'ท่าเรือด่านชายแดน ขาออกประเทศ' 'ท่าอากาศยานสุวรรณภูมิ ขาเข้าประเทศ'
 'ท่าอากาศยานสุวรรณภูมิ ขาออกประเทศ' 'ท่าอากาศยานดอนเมือง ขาเข้าประเทศ'
 'ท่าอากาศยานดอนเมือง ขาออกประเทศ'
 'ท่าอากาศยานอื่น ๆ ของ ทอท. ขาเข้าประเทศ'
 'ท่าอากาศยานอื่น ๆ ของ ทอท. ขาออกประเทศ'
 'ท่าอากาศยานภูมิภา

### ทำความสะอาดข้อมูล (Data Cleaning)

In [24]:
# Check null values
print('Null — passengers68.csv')
print(df68.isnull().sum())
print()
print('Null — passengers69.csv')
print(df69.isnull().sum())

Null — passengers68.csv
รูปแบบการเดินทาง     53744
วัตถุประสงค์         53744
สาธารณะ/ส่วนบุคคล    53744
หน่วยงาน             53744
ยานพาหนะ/ท่า         53744
วันที่               53744
หน่วย                53744
ปริมาณ               54052
dtype: int64

Null — passengers69.csv
รูปแบบการเดินทาง       0
วัตถุประสงค์           0
สาธารณะ/ส่วนบุคคล      0
หน่วยงาน               0
ยานพาหนะ/ท่า           0
วันที่                 0
หน่วย                  0
ปริมาณ               136
dtype: int64


In [25]:
# Check duplicates
dup68 = df68.duplicated().sum()
dup69 = df69.duplicated().sum()
print(f'Duplicates 68: {dup68:,} rows ({dup68/len(df68)*100:.1f}%)')
print(f'Duplicates 69: {dup69:,} rows')

Duplicates 68: 53,744 rows (77.4%)
Duplicates 69: 0 rows


In [26]:
# Check data types
print(df68.dtypes)
print()
print('Sample ปริมาณ:', df68['ปริมาณ'].head(3).tolist())

รูปแบบการเดินทาง     object
วัตถุประสงค์         object
สาธารณะ/ส่วนบุคคล    object
หน่วยงาน             object
ยานพาหนะ/ท่า         object
วันที่               object
หน่วย                object
ปริมาณ               object
dtype: object

Sample ปริมาณ: ['127,551', '8,218', '877,943']


In [27]:
# Cleaning Log — สรุปปัญหาที่พบก่อนทำความสะอาด

dup68  = df68.duplicated().sum()
null68 = df68.isnull().sum().sum()
dup69  = df69.duplicated().sum()
null69 = df69['ปริมาณ'].isnull().sum()

print('Issue                              | File              | Rows affected')
print('-' * 70)
print(f'Duplicate rows                     | passengers68.csv  | {dup68:,} ({dup68/len(df68)*100:.0f}%)')
print(f'ปริมาณ stored as string with comma | Both files        | All rows')
print(f'วันที่ stored as string            | Both files        | All rows')
print(f'Null in ปริมาณ                     | passengers69.csv  | {null69} rows')

Issue                              | File              | Rows affected
----------------------------------------------------------------------
Duplicate rows                     | passengers68.csv  | 53,744 (77%)
ปริมาณ stored as string with comma | Both files        | All rows
วันที่ stored as string            | Both files        | All rows
Null in ปริมาณ                     | passengers69.csv  | 136 rows


### จัดรูปแบบข้อมูลให้พร้อมสำหรับการวิเคราะห์

In [28]:
def clean_df(df, label):
    print(f'--- {label} ---')
    print(f'Before: {df.shape[0]:,} rows')

    # Remove duplicates
    df = df.drop_duplicates()
    print(f'After dedup: {df.shape[0]:,} rows')

    # Convert ปริมาณ to numeric
    df['ปริมาณ'] = df['ปริมาณ'].astype(str).str.replace(',', '').str.strip()
    df['ปริมาณ'] = pd.to_numeric(df['ปริมาณ'], errors='coerce')

    # Convert วันที่ to datetime
    df['วันที่'] = pd.to_datetime(df['วันที่'], format='%d/%m/%Y', errors='coerce')

    # Drop null
    n_null = df['ปริมาณ'].isnull().sum()
    df = df.dropna(subset=['ปริมาณ'])
    print(f'Dropped null ปริมาณ: {n_null} rows')

    # Add helper columns
    df['year']       = df['วันที่'].dt.year
    df['month']      = df['วันที่'].dt.month
    df['dayofweek']  = df['วันที่'].dt.dayofweek  # 0=Mon, 6=Sun
    df['is_weekend'] = df['dayofweek'] >= 5

    print(f'After clean: {df.shape[0]:,} rows')
    print()
    return df.reset_index(drop=True)

df68 = clean_df(df68, 'passengers68.csv')
df69 = clean_df(df69, 'passengers69.csv')

df_all = pd.concat([df68, df69], ignore_index=True)
print(f'Total: {df_all.shape[0]:,} rows')

--- passengers68.csv ---
Before: 69,440 rows
After dedup: 15,696 rows
Dropped null ปริมาณ: 309 rows
After clean: 15,387 rows

--- passengers69.csv ---
Before: 3,010 rows
After dedup: 3,010 rows
Dropped null ปริมาณ: 136 rows
After clean: 2,874 rows

Total: 18,261 rows


In [29]:
# Verify
print('Date range 68:', df68['วันที่'].min().date(), 'to', df68['วันที่'].max().date())
print('Date range 69:', df69['วันที่'].min().date(), 'to', df69['วันที่'].max().date())
print('Vehicle types:', df_all['ยานพาหนะ/ท่า'].nunique())
df68.head()

Date range 68: 2025-01-01 to 2025-12-31
Date range 69: 2026-01-01 to 2026-03-11
Vehicle types: 43


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ,year,month,dayofweek,is_weekend
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,2025-01-01,คน,127551.0,2025,1,2,False
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,2025-01-01,คน,8218.0,2025,1,2,False
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),2025-01-01,คัน,877943.0,2025,1,2,False
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),2025-01-01,คัน,932642.0,2025,1,2,False
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),2025-01-01,คัน,1364992.0,2025,1,2,False
